# ConvoKit Reddit Corpus Processing and EDA

In [1]:
from convokit import Corpus, download

import json
from pathlib import Path
import pandas as pd
import polars as pl
from tqdm import tqdm
import numpy as np
from csv import DictReader

from sklearn.model_selection import train_test_split
from pyprojroot import here

from score_usernames import main as score_usernames

In [2]:
subreddits = [
    "movies",
    "gardening",
    "booksuggestions",
    "Baking",
    "Fitness",
    "MechanicAdvice",
    "HomeImprovement",
    "explainlikeimfive",
    "askscience",
    "AskDocs",
    "NoStupidQuestions",
    "Cooking",
    "TravelHacks",
    "personalfinance",
    "netflix",
    "cars",
    "AskCulinary",
    "learnprogramming",
    "backpacking",
    "camping",
    "bodyweightfitness",
    "socialskills",
    "Fantasy",
    "cycling",
    "sleep",
    "CasualConversation",
    "AskMen",
    "hiking",
    "houseplants",
    "languagelearning",
    "Frugal",
    "photography",
    "selfimprovement",
    "Pets",
    "declutter",
    "Meditation",
    "running",
    "AskHistorians",
    "TooAfraidToAsk",
    "boardgames",
    "productivity",
]

downloads = dict()
corpora = dict()
for subreddit in subreddits:
    try:
        downloads[subreddit] = download(f"subreddit-{subreddit}")
    except:
        print(f"Failed to download {subreddit}")

Failed to download movies
Failed to download gardening
Failed to download booksuggestions
Failed to download Baking
Failed to download Fitness
Failed to download MechanicAdvice
Failed to download HomeImprovement
Failed to download explainlikeimfive
Failed to download askscience
Failed to download AskDocs
Failed to download NoStupidQuestions
Failed to download Cooking
Failed to download TravelHacks
Failed to download personalfinance
Failed to download netflix
Failed to download cars
Failed to download AskCulinary
Failed to download learnprogramming
Failed to download backpacking
Failed to download camping
Failed to download bodyweightfitness
Failed to download socialskills
Failed to download Fantasy
Failed to download cycling
Failed to download sleep
Failed to download CasualConversation
Failed to download AskMen
Failed to download hiking
Failed to download houseplants
Failed to download languagelearning
Failed to download Frugal
Failed to download photography
Failed to download selfimp

In [ ]:
for subreddit, corpus in corpora.items():
    print(f"Stats for r/{subreddit}:")
    corpus.print_summary_stats()
    print()

In [ ]:
speakers = dict()
for subreddit, download in downloads.items():
    with open(Path(download) / "users.json") as f:
        print(f"Loading r/{subreddit} users...")
        speakers[subreddit] = json.load(f)
        print("Done!\n")

In [ ]:
all_usernames = set()
for speaker_set in speakers.values():
    all_usernames.update(speaker_set.keys())

len(all_usernames)

In [ ]:
for user in all_usernames:
    if "maga" in user:
        print(user)

In [ ]:
with open(here("data/all_usernames.csv"), mode = "w") as f:
    f.write("username,score\n")
    for username in all_usernames:
        f.write(f"{username},0\n")

In [ ]:
score_usernames()

In [ ]:
users_scored = pd.read_csv(here("data/all_usernames_scored.csv"))
users_scored.sort_values(by="score", inplace=True)
users_scored.set_index("username", inplace=True)

In [ ]:
users_scored.head(50)

In [ ]:
# Determine common scores
users_scored["score"].value_counts()

In [ ]:
# Get a mapping from id to username for all posts

replies = []
for subreddit, download in downloads.items():
    with open(Path(download) / "utterances.jsonl") as f:
        posts = dict() # Dictionary mapping post id to poster username

        print(f"Loading r/{subreddit} posts...")
        for utterance_string in tqdm(f):
            utterance = json.loads(utterance_string)
            if utterance["id"] == utterance["root"]:
                posts[utterance["id"]] = utterance["user"]

        print(f"Loading r/{subreddit} replies...")
        f.seek(0)
        for utterance_string in tqdm(f):
            utterance = json.loads(utterance_string)
            if utterance["reply_to"] == utterance["root"]:
                replies.append({
                    "id": utterance["id"],
                    "subreddit": subreddit,
                    "username": posts[utterance["root"]],
                    "content": utterance["text"]
                })
        
        del posts
        print("Done!")
        print()
        

In [ ]:
# Assign scores and prune missing usernames
print(f"Started with {len(replies)} replies")
replies = [reply for reply in tqdm(replies) if reply["username"] != "[deleted]"]
print(f"Now have {len(replies)} replies")

for reply in tqdm(replies):
    reply["username_score"] = users_scored.at[reply["username"], "score"]

In [ ]:
# Write the replies to a file
with open(here("data/all_direct_replies.csv"), mode = "w") as f:
    f.write("id,subreddit,username,username_score,content\n")
    for reply in tqdm(replies):
        f.write(f"{reply["id"]},{reply["subreddit"]},{reply["username"]},{reply["username_score"]},{repr(reply["content"])}\n")

In [2]:
# Need to load into a single column and then manually split due to appearance of delimeter in last column
reply_df = pl.scan_csv(here("data/all_direct_replies.csv"), has_header=True, separator=chr(0), quote_char=None)

In [3]:
reply_df.head().collect()

"id,subreddit,username,username_score,content"
str
"""c02zwul,movies,Mendokusai,0.0,…"
"""c033ytz,movies,djolemcloud,0.0…"
"""c0343cf,movies,bighippo,0.0,'g…"
"""c0345dl,movies,ataraxis,0.0,'1…"
"""c034g4l,movies,qtoo,0.0,""Don't…"


In [4]:
# Split out the columns properly
header = reply_df.columns[0]
header_split = header.split(",")
reply_df = reply_df.select(
    pl.col(header)
    .str.splitn(",", len(header_split))
    .struct.rename_fields(header_split)
    .alias("fields")
).unnest("fields")

/var/folders/_b/7c8j58qs08x58wp8t3hfhcy40000gn/T/ipykernel_69037/210579771.py:2: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  header = reply_df.columns[0]


In [5]:
# Cast username score to float
reply_df = reply_df.with_columns(
    pl.col("username_score").cast(pl.Float64)
)

In [6]:
reply_df.head().collect()

id,subreddit,username,username_score,content
str,str,str,f64,str
"""c02zwul""","""movies""","""Mendokusai""",0.0,"""'The Borat one is a little ove…"
"""c033ytz""","""movies""","""djolemcloud""",0.0,"""'[deleted]'"""
"""c0343cf""","""movies""","""bighippo""",0.0,"""'great site for streaming vide…"
"""c0345dl""","""movies""","""ataraxis""",0.0,"""'1. There Will Be Blood 9.1 (1…"
"""c034g4l""","""movies""","""qtoo""",0.0,"""""Don't forget Ryan Vs. Dorkman…"


In [7]:
# Separate out into liberal, conservative, neutral, and unknown
reply_df = reply_df.with_columns(
    pl.when(pl.col("username_score") > 0.39)
    .then(pl.lit("conversative"))
    .when(pl.col("username_score") < -0.39)
    .then(pl.lit("liberal"))
    .when(abs(pl.col("username_score")) < 0.01)
    .then(pl.lit("neutral"))
    .otherwise(pl.lit("unknown"))
    .alias("label")
)

In [8]:
# Remove unknown labels
reply_df = reply_df.filter(pl.col("label") != "unknown")

In [10]:
idx = np.arange(reply_df.select(pl.col("id")).count().collect().row(0)[0])
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=88)

In [11]:
train_df = reply_df.with_row_index().filter(
    pl.col("index").is_in(train_idx)
)

test_df = reply_df.with_row_index().filter(
    pl.col("index").is_in(test_idx)
)

In [ ]:
# Write each dataset to file

to_write = [
    ("x_train.txt", train_df["content"].collect(), lambda s: s[1:-1]), # delete quotes
    ("y_train.txt", train_df["label"].collect(), lambda s: s),
    ("x_test.txt", test_df["content"].collect(), lambda s: s[1:-1]), # delete quotes
    ("y_test.txt", test_df["label"].collect(), lambda s: s)
]

for p, l, fun in to_write:
    with open(here(f"data/{p}"), mode = "w") as f:
        for e in tqdm(l):
            f.write(fun(e) + "\n")

In [13]:
train_df.sink_parquet(here("data/train.parquet"))
test_df.sink_parquet(here("data/test.parquet"))